In [5]:
print('ok')

ok


In [70]:
from typing import List, TypedDict, Literal,Dict
from pydantic import BaseModel, Field
import time
import re
import os
from pathlib import Path
from langchain_classic.chains import RetrievalQA



from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv

In [69]:
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone, ServerlessSpec

In [8]:
load_dotenv()

True

In [21]:
docs_d = PyPDFLoader(r"C:\Users\TPWODL\New folder_Content\Advanced_RAG_Project\data\raw\Supply Code, 2019 with Gazette.pdf").load()

In [22]:
print(len(docs_d))

95


In [23]:
print(type(docs_d))

<class 'list'>


In [26]:

# -----------------------------
# 0. INPUT NORMALIZATION (FIX)
# -----------------------------
def normalize_input(data):

    # Case 1: already string
    if isinstance(data, str):
        return data

    # Case 2: list of strings
    if isinstance(data, list):
        return "\n".join([str(x) for x in data])

    # Case 3: LangChain Documents
    try:
        from langchain.schema import Document
        if isinstance(data, list) and isinstance(data[0], Document):
            return "\n".join([doc.page_content for doc in data])
    except:
        pass

    # Case 4: dict/json
    if isinstance(data, dict):
        return str(data)

    raise ValueError("Unsupported input type for text cleaning")


# -----------------------------
# 1. CLEANING
# -----------------------------
def clean_text(text: str) -> str:

    text = re.sub(r"[‘’]", "'", text)
    text = re.sub(r'[“”]', '"', text)

    text = re.sub(r"\n\s*\d{1,4}\s*\n", "\n", text)
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)

    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


# -----------------------------
# 2. REMOVE NOISE
# -----------------------------
def remove_noise(text: str) -> str:
    lines = text.split("\n")

    freq = {}
    for line in lines:
        key = line.strip()
        if key:
            freq[key] = freq.get(key, 0) + 1

    cleaned = []
    for line in lines:
        if freq.get(line.strip(), 0) > 5:
            continue
        cleaned.append(line)

    return "\n".join(cleaned)


# -----------------------------
# 3. MAIN PIPELINE
# -----------------------------
def build_docs(raw_input):

    # ✅ FIX: normalize input
    text = normalize_input(raw_input)

    text = clean_text(text)
    text = remove_noise(text)

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=100
    )

    chunks = splitter.split_text(text)

    docs = [chunk.strip() for chunk in chunks if chunk.strip()]

    return docs



In [27]:
docs = build_docs(docs_d)



In [29]:
print(type(docs))

<class 'list'>


In [31]:
print(len(docs))

774


In [36]:
print(docs[3])

and such other matters. 
 
Odisha Electricity Regulatory Commission Distribution (Conditions of Supply) Code, 2019 
 
CHAPTER I 
 
 
1. SHORT TITLE, COMMENCEMENT 
1.1 This Code shall be called the "Odisha Electricity Regulatory Commission Distribution 
(Conditions of Supply) Code, 2019; herein after referred to as "Code". 
1.2 This Code shall come into force on the date of publication in the Official Gazette.


In [38]:
print(docs[4])

1.3 This Code shall extend to the whole of the State of Odisha.' metadata={'producer': 'GPL Ghostscript 9.06', 'creator': 'PScript5.dll Version 5.2', 'creationdate': '2019-10-19T13:51:32+05:30', 'author': 'spm', 'moddate': '2019-11-06T13:21:17+05:30', 'title': 'Microsoft Word - Supply Code, 2019 Final 19.10.2019', 'source': 'C:\\Users\\TPWODL\\New folder_Content\\Advanced_RAG_Project\\data\\raw\\Supply Code, 2019 with Gazette.pdf', 'total_pages': 95, 'page': 0, 'page_label': '1'}


In [39]:
print(docs[5])

page_content='2 
 
1.4 This Code shall be applicable to: 
(a) all Distribution and Retail Supply licensee/supplie rs including Deemed 
licensee/suppliers, all consumers, end users of electricity in the State of Odisha; 
(b) all other persons duly doing the business of distri bution/supply of electricity in his 
area of supply; 
(c) all other persons who are exempted under Section 13 of the Act; and 
(d) unauthorized supply, unauthorized use, diversion and other means of unauthorized


In [40]:
print(docs[6])

(d) unauthorized supply, unauthorized use, diversion and other means of unauthorized 
use/ abstraction of electricity. 
1.5 The Odisha General Clauses Act, 1937 shall apply to the interpretation of this Code. 
1.6 This Code shall supersede the Odisha Electricity Re gulatory Commission Distribution 
(Conditions of Supply) Code, 2004 and its subsequent amendments. 
1.7 On the application of the licensee/supplier(s) or s uo motu all forms and formats annexed


In [47]:
print(docs[7])

1.7 On the application of the licensee/supplier(s) or s uo motu all forms and formats annexed 
to this code may be suitably amended by the Commiss ion in the public interest by a


In [48]:
print(docs[8])

special order.' metadata={'producer': 'GPL Ghostscript 9.06', 'creator': 'PScript5.dll Version 5.2', 'creationdate': '2019-10-19T13:51:32+05:30', 'author': 'spm', 'moddate': '2019-11-06T13:21:17+05:30', 'title': 'Microsoft Word - Supply Code, 2019 Final 19.10.2019', 'source': 'C:\\Users\\TPWODL\\New folder_Content\\Advanced_RAG_Project\\data\\raw\\Supply Code, 2019 with Gazette.pdf', 'total_pages': 95, 'page': 1, 'page_label': '2'}
page_content='3 
 
CHAPTER II


In [55]:
print(docs[100])

or co-owner. The licensee/supplier may not supply power until such way-leave, licence, 
sanction or other right or interest is obtained. An y extra expenditure incurred in placing 
the service line in accordance with the terms of wa y leave, sanctioned licence or other 
right or interest obtained from the owner or co-own er shall be borne by the applicant. No 
way-leave, license, sanction or other right or inte rest once granted shall be cancelled or


In [57]:
documents = [Document(page_content=text) for text in docs]

In [58]:
documents

[Document(metadata={}, page_content="page_content='1 \n \n \n \n \n \n \n \n \n \n \n \n \n \n \nODISHA ELECTRICITY REGULATORY COMMISSION \nBHUBANESWAR – 751 021 \n***** \n \nNOTIFICATION \nThe 27 th August, 2019 \n \nNo.OERC/Engg/92/2003(Vol-VII)/1210 - In exercise of the powers conferred by Section \n50 read with and Section 181 (2) (t), (v), (w) and (x) read with Part-VI of the Electricity Act, \n2003 (Act 36 of 2003) and all other powers enabling it in this behalf, the Odisha Electricity"),
 Document(metadata={}, page_content='2003 (Act 36 of 2003) and all other powers enabling it in this behalf, the Odisha Electricity \nRegulatory Commission hereby make the following Sup ply Code by Notification to govern \nsupply of electricity by the licensee/supplier to t he consumers / end users and measures for \nrecovery of electricity charges, intervals for bill ing of electricity charges, disconnection of \nsupply of electricity for non-payment thereof, rest oration of supply of electricit

In [59]:
print(len(documents))

774


In [54]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 3. Initialize Pinecone client
api_key = os.getenv("PINECONE_API_KEY")
if not api_key:
    raise ValueError("PINECONE_API_KEY not found. Check your .env file!")

pc = Pinecone(api_key=api_key)
index_name = "rag-vector-project"

In [60]:
# 5. Create index with correct 1536 dimensions
if index_name not in pc.list_indexes().names():
    print(f"Creating new index: {index_name}")
    pc.create_index(
        name=index_name,
        dimension=1536, 
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

# 6. Connect to index and store documents
vector_store = PineconeVectorStore.from_documents(
    documents=documents,
    embedding=embeddings,
    index_name=index_name
)



In [61]:
# 7. Create retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 4})
print("Success! Documents are stored and retriever is ready.")

Success! Documents are stored and retriever is ready.


In [62]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [78]:
query1 = "what is Shifting of Meter / Existing Connection Code ?"

In [79]:
retriv_docs = retriever.invoke(query1)

In [80]:
retriv_docs

[Document(id='2df3cf15-5e27-424d-9b5c-85888b3621c9', metadata={}, page_content='Low Tension to High Tension or vice-versa, and from single-phase to three-phase or \nvice-versa, within the time limits as specified in the Standard of Performance Regulations \nfor respective class of consumers. \nShifting of Meter / Existing Connection \n45. No consumer can shift the meter without the knowled ge and consent of the \nlicensee/supplier. However, the consumer can apply for shifting the service \nconnection/meter in existing premises in the format prescribed in Form No.1 or 2 to this'),
 Document(id='13a3a0ec-5d8d-453a-b56a-860ef3b317fe', metadata={}, page_content="page_content='20 \n \nstated in the Commission's approval. If the plan is made by government the same shall be \nincluded by the licensee/supplier in the programme of electrification. \n Shifting of service connection/deviation of lines and shifting of equipment \n37. Wherever the consumers request for shifting the ser vice connect

In [81]:
for i, doc in enumerate(retriv_docs):
    print(f"\nResult {i+1}:")
    print(doc.page_content)


Result 1:
Low Tension to High Tension or vice-versa, and from single-phase to three-phase or 
vice-versa, within the time limits as specified in the Standard of Performance Regulations 
for respective class of consumers. 
Shifting of Meter / Existing Connection 
45. No consumer can shift the meter without the knowled ge and consent of the 
licensee/supplier. However, the consumer can apply for shifting the service 
connection/meter in existing premises in the format prescribed in Form No.1 or 2 to this

Result 2:
page_content='20 
 
stated in the Commission's approval. If the plan is made by government the same shall be 
included by the licensee/supplier in the programme of electrification. 
 Shifting of service connection/deviation of lines and shifting of equipment 
37. Wherever the consumers request for shifting the ser vice connection or for deviation for 
the existing lines at their cost the following time schedule shall be observed for

Result 3:
the existing lines at their cost

In [63]:
query = "What is Processing of Applications ?"

In [64]:
retriv_docs = retriever.invoke(query)

In [65]:
retriv_docs

[Document(id='f510108d-673f-46cf-b543-bb1bf778048b', metadata={}, page_content='within 3 days from the date of receipt of application regard ing shortcomings in \nthe application form if any. \n(ii) The licensee/supplier shall maintain a per manent record of all application forms \nreceived in an Application Register/Database. Each application form shall be \nallotted a permanent application number (for identi fication) serially in the order in \nwhich it was received. Separate registers/databases for different category of'),
 Document(id='76d98b10-b0d9-437e-a199-e8d035011e48', metadata={}, page_content='which it was received. Separate registers/databases for different category of \nconsumers may be maintained. The licensee/supplier shall keep the registers/ \ndatabases updated with stage-wise status of disposa l of each application form and \ndisplay in its website. \n(iii) The licensee/supplier shall deal with applic ation forms in each tariff category on \nthe broad principle of "fi

In [66]:
for i, doc in enumerate(retriv_docs):
    print(f"\nResult {i+1}:")
    print(doc.page_content)


Result 1:
within 3 days from the date of receipt of application regard ing shortcomings in 
the application form if any. 
(ii) The licensee/supplier shall maintain a per manent record of all application forms 
received in an Application Register/Database. Each application form shall be 
allotted a permanent application number (for identi fication) serially in the order in 
which it was received. Separate registers/databases for different category of

Result 2:
which it was received. Separate registers/databases for different category of 
consumers may be maintained. The licensee/supplier shall keep the registers/ 
databases updated with stage-wise status of disposa l of each application form and 
display in its website. 
(iii) The licensee/supplier shall deal with applic ation forms in each tariff category on 
the broad principle of "first come, first served" b asis as per serial priority in the

Result 3:
connection shall be treated as separate one and all the relevant Regulations as

In [71]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff"
)

response = qa_chain.invoke({"query": query})



In [74]:
print(response["result"] + "\n")


Processing of Applications refers to the procedure followed by a licensee or supplier when handling application forms for the release of supply to new connections. This includes verifying the application form and the enclosed documents. If any deficiencies are found, the licensee/supplier is required to issue a written note to the applicant regarding the shortcomings within 3 days from the date of receipt of the application. The process also involves maintaining a permanent record of all application forms, allotting a permanent application number for identification, and updating the status of each application in a register or database. Applications are dealt with on a "first come, first served" basis according to their serial priority.



In [ ]:
query_01 = "what is Shifting of Meter / Existing Connection Code?"

In [76]:
response_01 = qa_chain.invoke({"query": query_01})


In [77]:
print(response_01["result"] + "\n")


The Shifting of Meter / Existing Connection Code refers to the regulations and procedures that govern the process of shifting a meter or service connection for consumers. According to the provided context, no consumer can shift the meter without the knowledge and consent of the licensee/supplier. Consumers can apply for shifting the service connection/meter using a prescribed format (Form No.1 or 2). 

Additionally, there are specific time schedules for completing the shifting of various types of connections, which include:
- Shifting of meter/service: 15 days
- Shifting of LT lines: 30 days
- Shifting of 11KV lines: 60 days
- Shifting of 33KV lines: 90 days
- Shifting of 33/11 KV Distribution Transformer structures: 90 days

These timeframes include the time required for preparation of estimates, collection of deposits, and execution of the work.



In [82]:
query_02 = "What is a new service connection, how can I apply, and what are the rules and regulations?"

In [83]:
response_02 = qa_chain.invoke({"query": query_02})


In [84]:
print(response_02["result"] + "\n")


A new service connection refers to the process of establishing a new electrical supply to a premises that does not currently have an electrical connection. 

To apply for a new service connection, you typically need to fill out an application form (Form No. 1) addressed to the Junior Engineer or Sub-Divisional Engineer of the relevant distribution licensee. The application must be signed by the owner or lawful occupier of the premises, with the owner's consent if applicable.

The rules and regulations for applying for a new service connection include:

1. Providing identity proof of the applicant.
2. Submitting proof of ownership or legal occupancy of the premises for which the connection is sought.
3. Providing proof of the applicant's current address.
4. Ensuring that the application is duly filled and signed.

Additionally, the licensee/supplier will verify the application and the enclosed documents. If there are any deficiencies, they will issue a written note to the applicant rega